This notebook provides a guided tour of the codebase used to replicate Table 1 of Kelly & Pruitt (2013). The core of this analysis relies on constructing a single predictor (a latent factor) from a large cross-section of assets to forecast market returns.

We will walk through:

Data Ingestion & Cleaning: How we handle the raw Fama-French data.

Data Sparsity: Visualizing the availability of our cross-sectional predictors (the 6, 25, and 100 portfolios).

The Three-Pass Regression Filter: Visualizing the intermediate steps of the target-proxy algorithm.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from pathlib import Path

# Import our custom modules
import load_data
import replication
from settings import config

OUTPUT_DIR = Path(config("OUTPUT_DIR"))

## Data Ingestion & Cleaning
The raw Fama-French datasets contain missing value indicators such as -99.99, -999, -999.00, and -99.990. Our load_data.py script standardizes these to NaN and calculates the Monthly Book-to-Market (BM) ratios.

We apply a log transformation to the BM ratios, matching the methodology specified by Vuolteenaho.

In [ ]:
# Load the cleaned data from our cache
print("Loading cleaned datasets...")
data = load_data.clean_kelly_pruitt_data(load_from_cache=True)

bm_25 = data["25_Portfolios_5x5_BM"]
print(f"Data spans from {bm_25.index.min().date()} to {bm_25.index.max().date()}")

## Analyzing Data Sparsity
When dealing with large cross-sections (like the 100 Portfolios formed on Size and Book-to-Market), not all portfolios have data going back to the beginning of the sample. To understand our cross-sectional sample size over time, we can plot the number of non-null portfolios available in each dataset for every month.

In [ ]:
# Run sparsity analysis (generates and saves the plot via replication.py)
valid_6, valid_25, valid_100 = replication.data_sparsity_analysis(data)

# Display the generated image inline
display(Image(filename=OUTPUT_DIR / "data_sparsity.png"))

## Summary Statistics


In [ ]:
summary_df = replication.generate_summary_statistics(data)
display(summary_df)

## The Three-Pass Regression Filter (Intermediate Steps)
The magic of the Kelly & Pruitt (2013) approach is extracting a single predictive factor, $F_t$, from the cross-section of Book-to-Market ratios.

Let's use the 25 Portfolios and a 1-month forecast horizon ($h=1$) to visualize the intermediate outputs of regression_tools.py.

In [ ]:
# Define our target variable (1-Month Log Market Returns)
y_1m = data['Market_Returns']['Log_Mkt']
v_df = data["25_Portfolios_5x5_BM"]  # The cross-section of predictors

# Align data indices
common_idx = v_df.index.intersection(y_1m.index)
v_df_aligned = v_df.loc[common_idx]
y_1m_aligned = y_1m.loc[common_idx]

### Stage 1: Time-Series Regressions (Estimating Sensitivities)

First, we run a time-series regression for each portfolio $i$ to estimate its sensitivity to future market returns.

We regress the portfolio's characteristic $v_{i,t}$ on the future market return $y_{t+h}$:$$v_{i,t} = \phi_{i,0} + \phi_i y_{t+h} + e_{i,t}$$

In [ ]:
# Run Stage 1
phi = replication.run_stage_1_analysis(v_df_aligned, y_1m_aligned)

# Display the Stage 1 plot
display(Image(filename=OUTPUT_DIR / "stage_1_sensitivities.png"))

### Stage 2: Cross-Sectional Regressions (Extracting the Factor)

Next, at each time step $t$, we run a cross-sectional regression of the portfolio characteristics $v_{i,t}$ onto their estimated sensitivities $\phi_i$:$$v_{i,t} = c_t + F_t \phi_i + w_{i,t}$$

In [ ]:
# Run Stage 2
F_series = replication.run_stage_2_analysis(v_df_aligned, phi)

# Display the Stage 2 plot
display(Image(filename=OUTPUT_DIR / "stage_2_factor.png"))

### Stage 3: Predictive Regression

Finally, we regress future market returns on our newly extracted, lagged factor $F_t$ to build our prediction model.$$y_{t+h} = \beta_0 + \beta F_t + u_{t+h}$$

In [ ]:
# Run Stage 3
model = replication.run_stage_3_analysis(F_series, y_1m_aligned)

# Display the Stage 3 plot
display(Image(filename=OUTPUT_DIR / "stage_3_predictive.png"))